In [44]:
import csv
import pandas as pd

In [65]:
# load target artists from csv with their id lookup
target_artists_df = pd.read_csv("./artists.csv")
target_artists_df

,id,name,birth_year,death_year,exhibitions,mixed?
0,0,Isamu Noguchi,1904,1988.0,"[5, 10]",NaN
1,1,Amanda Ross-Ho,1975,NaN,[1],NaN
2,2,Laurel Nakadate,1975,NaN,"[1, 13]",NaN
3,3,Mequitta Ahuja,1976,NaN,[1],NaN
4,4,Kip Fulbeck,1965,NaN,"[0, 1]",NaN
5,5,Ruth Asawa,1926,2013.0,"[2, 10]",no
6,6,Do Ho Suh,1962,NaN,[4],no
7,7,Oscar yi Hou,1998,NaN,[6],NaN
8,8,Sarah Sze,1969,NaN,[7],NaN
9,9,Wu Tsang,1982,NaN,[10],NaN


In [66]:
artist_ids = {}
artist_bio = {}

with open("../opendata/data/constituents.csv", newline='', encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row["forwarddisplayname"].strip().lower()

        if name in target_artists_df["name"].str.lower().values:
            cid = row["constituentid"]

            artist_ids[cid] = name
            artist_bio[cid] = {}
            # add all rows to the bio dict for later use
            for key in row:
                if key != "constituentid":
                    row[key] = row[key].strip()
                    artist_bio[cid][key] = row[key]
            

In [67]:
artist_ids

{'2213': 'isamu noguchi',
 '2668': 'alfonso ossorio',
 '2924': 'ruth asawa',
 '20494': 'martin wong',
 '29899': 'hung liu',
 '33095': 'byron kim',
 '34317': 'do ho suh'}

In [68]:
artist_bio

{'2213': {'uuid': '8e8e560d-b6d1-4884-974e-fa893c8bcb8e',
  'ulanid': '500008602',
  'preferreddisplayname': 'Noguchi, Isamu',
  'forwarddisplayname': 'Isamu Noguchi',
  'lastname': 'Noguchi',
  'displaydate': 'American, 1904 - 1988',
  'artistofngaobject': '1',
  'beginyear': '1904',
  'endyear': '1988',
  'visualbrowsertimespan': '1901 to 1925',
  'nationality': 'American',
  'visualbrowsernationality': 'American',
  'constituenttype': 'individual',
  'wikidataid': 'Q442628'},
 '2668': {'uuid': '07dc6f3b-dfb1-44f9-82b6-806e45a13b5e',
  'ulanid': '500026802',
  'preferreddisplayname': 'Ossorio, Alfonso',
  'forwarddisplayname': 'Alfonso Ossorio',
  'lastname': 'Ossorio',
  'displaydate': 'American, born Philippines, 1916 - 1990',
  'artistofngaobject': '1',
  'beginyear': '1916',
  'endyear': '1990',
  'visualbrowsertimespan': '1901 to 1925',
  'nationality': 'American',
  'visualbrowsernationality': 'American',
  'constituenttype': 'individual',
  'wikidataid': 'Q827547'},
 '2924': {

In [69]:
object_ids = set()
object_to_artist = {}

with open("../opendata/data/objects_constituents.csv", newline='', encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        cid = row["constituentid"]

        if cid in artist_ids and row["roletype"] == "artist":
            oid = row["objectid"]

            object_ids.add(oid)
            object_to_artist[oid] = cid

In [70]:
len(object_ids)

82

In [71]:
target_artists_df.columns

Index(['id', 'name', 'birth_year', 'death_year', 'exhibitions', 'mixed?'], dtype='str')

In [91]:
artworks = []
sample = []

with open("../opendata/data/objects.csv", newline='', encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        oid = row["objectid"]

        if oid in object_ids:
            cid = object_to_artist[oid]

            # lookup_id is the id of the artist in the target artists df that we can link through their name
            artist_name = artist_ids[cid]
            filtered_names = target_artists_df[target_artists_df["name"].str.lower() == artist_name]
            lookup_id = int(filtered_names.iloc[0]["id"])

            artworks.append({
                "id": oid,
                "artist": lookup_id,
                "title": row["title"],
                "alt_text": "",
                "description": "",
                "date_created": row["displaydate"],
                "date_acquired_or_updated": row["accessionnum"][0:4],
                "institution": "7",
                "medium": row["medium"] + " | " + row["classification"],
                "image_url": ""
            })

In [92]:
artworks[:5]

[{'id': '56100',
  'artist': 0,
  'title': 'Great Rock of Inner Seeking',
  'alt_text': '',
  'description': '',
  'date_created': '1974',
  'date_acquired_or_updated': '1976',
  'institution': '7',
  'medium': 'basalt | Sculpture',
  'image_url': ''},
 {'id': '57425',
  'artist': 21,
  'title': 'Perpetual Sacrifice',
  'alt_text': '',
  'description': '',
  'date_created': '1949',
  'date_acquired_or_updated': '1979',
  'institution': '7',
  'medium': 'ink, wax, and watercolor on Whatman watercolor board | Drawing',
  'image_url': ''},
 {'id': '57731',
  'artist': 5,
  'title': 'Chrysanthemums',
  'alt_text': '',
  'description': '',
  'date_created': '1965',
  'date_acquired_or_updated': '1980',
  'institution': '7',
  'medium': 'color lithograph | Print',
  'image_url': ''},
 {'id': '61633',
  'artist': 5,
  'title': 'Flowers I (Title Page)',
  'alt_text': '',
  'description': '',
  'date_created': '1965',
  'date_acquired_or_updated': '1983',
  'institution': '7',
  'medium': 'lith

In [93]:
# get number of artworks for each artist
artist_counts = {}
for artwork in artworks:
    artist = artwork["artist"]
    if artist not in artist_counts:
        artist_counts[artist] = 0
    artist_counts[artist] += 1

print(artist_counts)
print(len(artworks))

{0: 5, 21: 14, 5: 55, 10: 2, 29: 3, 12: 2, 6: 1}
82


In [94]:
id_to_image = {}
id_to_alt_text = {}

with open("../opendata/data/published_images.csv", newline='', encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        id_to_image[row["depictstmsobjectid"]] = row["iiifthumburl"]
        id_to_alt_text[row["depictstmsobjectid"]] = row["assistivetext"]


for artwork in artworks:
    artwork["image_url"] = id_to_image.get(artwork["id"], None)
    artwork["alt_text"] = id_to_alt_text.get(artwork["id"], None)

print(artworks[0])

{'id': '56100', 'artist': 0, 'title': 'Great Rock of Inner Seeking', 'alt_text': 'A rough-hewn block of basalt sits upright on a rectangular stone base on a pale pink stone floor against a stone wall. In this photograph, we are off the front left corner of the free-standing sculpture in a space filled with natural light. The tongue-shaped basalt is narrower from front to back and wider from side to side. It is about three times taller than it is wide at its widest. Some of the dark brown basalt running up the center of the block is smooth but most is pitted. Strips up either side have been carved to expose rough, copper-colored stone, some of which has flaked away. The tan stone base is speckled densely with gray, and the leaves of an indoor but mature tree bow into the top left corner of the photograph.', 'description': '', 'date_created': '1974', 'date_acquired_or_updated': '1976', 'institution': '7', 'medium': 'basalt | Sculpture', 'image_url': 'https://api.nga.gov/iiif/20e46356-82d

In [95]:
# print any artworks that don't have an image url
for artwork in artworks:
    if artwork["image_url"] is None:
        print(artwork)

{'id': '57731', 'artist': 5, 'title': 'Chrysanthemums', 'alt_text': None, 'description': '', 'date_created': '1965', 'date_acquired_or_updated': '1980', 'institution': '7', 'medium': 'color lithograph | Print', 'image_url': None}
{'id': '61634', 'artist': 5, 'title': 'Flowers', 'alt_text': None, 'description': '', 'date_created': '1965', 'date_acquired_or_updated': '1983', 'institution': '7', 'medium': 'portfolio w/ 12 color lithographs (stone) including title page and colophon on German Etching paper | Portfolio', 'image_url': None}
{'id': '61648', 'artist': 5, 'title': 'Desert Flower', 'alt_text': None, 'description': '', 'date_created': '1965', 'date_acquired_or_updated': '1983', 'institution': '7', 'medium': 'lithograph (stone) in black on German Etching paper | Print', 'image_url': None}
{'id': '61649', 'artist': 5, 'title': 'Desert Flower', 'alt_text': None, 'description': '', 'date_created': '1965', 'date_acquired_or_updated': '1983', 'institution': '7', 'medium': 'lithograph (s

In [96]:
# export artworks and artist dfs
artworks_df = pd.DataFrame(artworks)
artists_df = pd.DataFrame.from_dict(artist_bio, orient="index")
artworks_df.to_csv("./exports/nga/artworks.csv", index=False)
artists_df.to_csv("./exports/nga/artists.csv", index=False)